# Lesson 3 : Session (Conversation Thread)

By using session in Agent Framework, you can handle the conversation thread in agent.  
Unlike in Lesson 6, this is a **short-term memory** that holds a single volatile thread of conversation.

In this exercise, we'll learn how to handle session.

## Prepare agent

Same as previous exercise, create an agent to connect to Microsoft Foundry.

In [1]:
from dotenv import load_dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity.aio import AzureCliCredential
from agent_framework import Agent, tool
from typing import Annotated
from pydantic import Field
from random import randint

load_dotenv()

# initialize a client object
credential = AzureCliCredential()
client = FoundryChatClient(credential=credential)

# define local tools
@tool(approval_mode="never_require")
def get_weather(
    location: Annotated[str, Field(description="the location to get the weather for")],
) -> str:
    """Get the weather for a given location."""
    conditions = ["sunny", "cloudy", "rainy", "stormy"]
    return f"The weather in {location} is {conditions[randint(0, 3)]}."

@tool(approval_mode="never_require")
def get_temperature(
    location: Annotated[str, Field(description="the location to get the temperature for")],
) -> str:
    """Get the temperature for a given location."""
    return f"The temperature in {location} is {randint(10, 30)} degrees."

# connect to the agent
agent = Agent(
    name="BasicWeatherAgent",
    client=client,
    instructions="You are an agent about weather information.",
    tools=[get_weather, get_temperature])

## Run agent without thread

First, let's run the following example without session (without conversation thread).<br>
In this example, first we ask weather and temperature, and next we ask to create a weather summary.

In Agent Framework, each request runs statelessly by default, and these 2 requests are performed in different agent thread.<br>
As a result, you will find that **the responses are inconsistent**.

> Note : In this execution (without session), the response id in Azure OpenAI Responses API is not inherited internally. (Tool execution is also called twice.) Use tracing and see the internal steps.

In [2]:
from IPython.display import Markdown, display

result = await agent.run("Tell me the weather and temperature in Osaka.")
display(Markdown(result.text))

result = await agent.run("Write a summary of the weather in Osaka. In the summary, include two things: the general weather conditions and important points to note.")
display(Markdown(result.text))

Osaka is currently **stormy**, with a temperature of **10 °C**.

### General weather conditions (Osaka)
- Cloudy skies with a temperature around **25°C**.

### Important points to note
- With **cloud cover**, visibility and sunlight may be reduced; conditions can feel a bit muted compared with clear weather.
- At **~25°C**, it may feel **mild to warm**, so light, breathable clothing is typically comfortable.

## Run agent with session (thread)

Now we fix this code as follows.<br>
By assigning same ```AgentSession```  object in ```run()``` method, the agent will run on the same conversation thread. (In this execution, the response id in Azure OpenAI Responses API will be inherited internally. The internal implementation depends on your using client.)

As a result, you can see the consistent response.

In [3]:
session = agent.create_session()

result = await agent.run(
    "Tell me the weather and temperature in Osaka.",
    session=session,
)
display(Markdown(result.text))

result = await agent.run(
    "Write a summary of the weather in Osaka. In the summary, include two things: the general weather conditions and important points to note.",
    session=session,
)
display(Markdown(result.text))

Osaka is **cloudy**, and the temperature is **14°C**.

**Osaka weather summary:**  
- **General conditions:** Cloudy with a temperature around **14°C**.  
- **Important points to note:** Limited sunshine and potentially muted visibility; consider a light layer for the cool conditions.